# GRAG downstream: ExplaGraphs classification on Colab

**Input:** a completed, checksummed Kaggle compact preprocessing backup in Google Drive.
No sentence encoder, ego-graph retrieval, or preprocessing needs to run again.

**Model:** `meta-llama/Llama-2-7b-chat-hf` with NF4 4-bit QLoRA.
Train LoRA adapters **and** the repository's GAT graph encoder/projector.
The quantized Llama backbone stays frozen.

**Output:** resumable checkpoints, raw predictions, metrics and experiment settings,
automatically copied to Drive. The notebook starts with a small smoke run.

This is a T4-conscious adaptation, **not an exact paper reproduction**.
Chat weights, quantization, prompt limits, training duration and preprocessing
choices must be recorded when comparing results.


## 1. Before starting

1. Select **Runtime > Change runtime type > T4 GPU**.
2. Accept the Llama-2 model license on Hugging Face using your own account.
3. Add a read-access token named **HF_TOKEN** in Colab Secrets.
   Enable notebook access. Never paste the token into a cell.
4. Ensure the updated repository, including `src/utils/notebook_downstream/`,
   has been pushed or uploaded before cloning it here.
5. Your preprocessing snapshot must contain **manifest.json** and every archive part.

We reuse Colab's installed PyTorch/CUDA. No isolated environment, `torch_scatter`,
`wandb`, or author-era package stack is required.


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime, then rerun this cell.")
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; memory: {gpu.total_memory / 2**30:.2f} GiB")
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

REPO_URL = "https://github.com/MichaelPham2005/BioGRAG-QA.git"
REPO_BRANCH = "GRAG/reproduce/WebQSP"
ROOT = Path("/content/BioGRAG-QA")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH,
                    "--single-branch", REPO_URL, str(ROOT)], check=True)
if not (ROOT / "src/utils/notebook_downstream/training.py").is_file():
    raise FileNotFoundError("Updated downstream helpers are missing. Push/upload them first.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


GPU: Tesla T4; memory: 14.56 GiB
PyTorch: 2.14.1+cu130 CUDA: 13.0


## 2. Install only missing dependencies

Do not reinstall PyTorch or CUDA.
If a package was already imported before installation and Colab requests a restart,
restart the session and rerun from section 1.


In [ ]:
requirements = {
    "torch_geometric": "torch-geometric",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "peft": "peft",
    "sentencepiece": "sentencepiece",
    "pandas": "pandas",
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
from importlib.metadata import version
print({name: version(name) for name in requirements.values()})

from src.utils.notebook_downstream import artifacts, training


{'torch-geometric': '2.8.1', 'transformers': '5.22.1', 'accelerate': '1.15.0', 'bitsandbytes': '0.50.2', 'peft': '0.21.2', 'sentencepiece': '0.2.2', 'pandas': '2.2.3'}


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 3. Mount Drive and select a completed expla_graphs snapshot

The default folder is the one created by the Kaggle backup helper.
If you have several snapshots, copy the desired folder name into `SNAPSHOT_NAME`.
A missing manifest indicates an incomplete upload; do not restore that folder.

Archive and file sizes/checksums are verified.
Links, unexpected files and path traversal are rejected.
Restored paths are local Colab paths; producer hashes and configuration are
preserved without rewriting the old Kaggle paths.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DATASET = "expla_graphs"
BACKUP_ROOT = Path("/content/drive/MyDrive/GRAG preprocessing backups")
SNAPSHOT_NAME = ""  # Set this explicitly when more than one matching snapshot exists.
candidates = []
for path in sorted(BACKUP_ROOT.glob("*/manifest.json")):
    if json.loads(path.read_text()).get("dataset") == DATASET:
        candidates.append(path.parent)
print("Completed snapshots:", [path.name for path in candidates])
if SNAPSHOT_NAME:
    selected = [path for path in candidates if path.name == SNAPSHOT_NAME]
else:
    selected = candidates
if len(selected) != 1:
    raise ValueError("Set SNAPSHOT_NAME to exactly one of the listed folders, then rerun.")
SNAPSHOT = selected[0]
PREPARED, PREPROCESS_RUN, SNAPSHOT_HASH = artifacts.restore_snapshot(
    SNAPSHOT, ROOT / "restored" / DATASET / SNAPSHOT.name, DATASET,
)
dataset = artifacts.CompactTrainingDataset(PREPARED, PREPROCESS_RUN)
original_splits = dataset.get_idx_split()
print("Retained split counts:", {key: len(ids) for key, ids in original_splits.items()})
print("Excluded source rows:", dataset.metadata.get("excluded_count", 0))
print("Prepared:", PREPARED)
print("Compact samples:", PREPROCESS_RUN)


Mounted at /content/drive
Completed snapshots: ['expla_graphs-0af742e8d573e88c57f5']
Verified snapshot: expla_graphs, 2766 retained samples.
Retained split counts: {'train': 1659, 'val': 553, 'test': 554}
Excluded source rows: 0
Prepared: /content/BioGRAG-QA/restored/expla_graphs/expla_graphs-0af742e8d573e88c57f5/output/expla_graphs-prepared
Compact samples: /content/BioGRAG-QA/restored/expla_graphs/expla_graphs-0af742e8d573e88c57f5/output/expla_graphs-retrieval-fp32


## 4. Configure an experiment

- Leave `SMOKE_TEST=True` for the first run: up to 8 train / 4 validation / 4 test examples.
- After that works, set **SMOKE_TEST=False** and rerun from this section.
  The default run name changes, so smoke checkpoints cannot contaminate the full run.
- Full mode uses the original retained split membership and **3 epochs** by default,
  not the repository's 10-epoch setting.
- Microbatch 1 and accumulation 4 give an effective batch of up to 4.
  The final short group is normalized by its actual microbatch count.
- Use a new `RUN_NAME` after changing settings, input snapshot, or helper code.

Drive backup is always enabled here. An interrupted runtime can resume at the last
backed-up optimizer-step boundary (default every 50 updates), or completed epoch.
Keep the same snapshot, run name and settings when resuming.


In [ ]:
SMOKE_TEST = False
RUN_NAME = "expla-graphs-chat-qlora-" + ("smoke-v1" if SMOKE_TEST else "full-v2")
OUTPUT = ROOT / "output" / "downstream" / DATASET / RUN_NAME
DRIVE_OUTPUT = Path("/content/drive/MyDrive/GRAG downstream") / DATASET / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
if not (OUTPUT / "run.json").exists():
    artifacts.restore_training_outputs(DRIVE_OUTPUT, OUTPUT)

limits = {"train": 8, "val": 4, "test": 4}
splits = {name: list(ids[:limits[name]] if SMOKE_TEST else ids)
          for name, ids in original_splits.items()}
saved_run = json.loads((OUTPUT / "run.json").read_text()) if (OUTPUT / "run.json").exists() else None
print("Selected split counts:", {key: len(ids) for key, ids in splits.items()})
print("Drive output folder:", DRIVE_OUTPUT)


Selected split counts: {'train': 1659, 'val': 553, 'test': 554}
Drive output folder: /content/drive/MyDrive/GRAG downstream/expla_graphs/expla-graphs-chat-qlora-full-v2


## 5. Pin and load the chat model

**Why chat?** It is instruction-tuned and is explicitly named in the paper's
no-retrieval baseline comparison. This alone does not establish that every
experiment used that checkpoint. The repository's default is the base model;
this notebook deliberately selects the chat model.

**What learns?**

- LoRA rank **8**, alpha **16**, dropout **0.05**, targeting `q_proj` and `v_proj`.
- GAT: **4 layers, 4 heads, hidden width 1024** with query-conditioned scaling.
- Projector: **1024 -> 2048 -> Llama hidden width**, producing one soft graph token.

QLoRA adds adapters to frozen NF4 weights; it does not fully fine-tune 7 billion
parameters. Non-reentrant checkpointing reduces activation memory while keeping
the gradient path through Llama to the graph prompt.

The instruction delimiters follow the repository's `[INST] ... [/INST]` layout.
The graph token precedes the description and question inside that instruction.
Only answer/EOS tokens receive training labels; test generation receives no gold answer.


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi
import random

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets and enable notebook access.")
MODEL_ID = "meta-llama/Llama-2-7b-chat-hf"
MODEL_REVISION = ""  # Optional commit SHA; otherwise pin main once, or reuse the saved SHA.
requested_revision = MODEL_REVISION or (
    saved_run["settings"]["model_revision"] if saved_run else "main"
)
resolved_revision = HfApi(token=HF_TOKEN).model_info(
    MODEL_ID, revision=requested_revision,
).sha
settings = training.Settings(
    dataset=DATASET, model_id=MODEL_ID, model_revision=resolved_revision,
    epochs=1 if SMOKE_TEST else 10, batch_size=1, grad_accum=4,
    max_sequence_tokens=512, max_description_tokens=256,
    max_answer_tokens=8, patience=5
)
SIGNATURE = training.initialize_run(
    settings, dataset, splits, SNAPSHOT_HASH, OUTPUT, DRIVE_OUTPUT,
)
random.seed(settings.seed)
torch.manual_seed(settings.seed)
torch.cuda.manual_seed_all(settings.seed)
model = training.load_qlora(settings, HF_TOKEN)
llm_trainable, llm_total = model.llm.get_nb_trainable_parameters()
graph_trainable = sum(p.numel() for p in model.graph_encoder.parameters() if p.requires_grad)
projector_trainable = sum(p.numel() for p in model.projector.parameters() if p.requires_grad)
print("Pinned model revision:", resolved_revision)
print({"LLM logical parameters (includes adapters)": llm_total,
       "trainable LoRA": llm_trainable, "trainable GAT": graph_trainable,
       "trainable projector": projector_trainable})
print("HF_TOKEN is not included in the saved experiment/checkpoints.")


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Pinned model revision: f5db02db724555f92da89c216ac04704f23d4590
{'LLM logical parameters (includes adapters)': 6742609920, 'trainable LoRA': 4194304, 'trainable GAT': 29415426, 'trainable projector': 10491904}
HF_TOKEN is not included in the saved experiment/checkpoints.


## 6. Check text limits before training

Context budget: **512 tokens**, including the graph token and answer reserve.
Descriptions are capped at 256 tokens; questions are never silently truncated.
Over-budget questions raise an error. Answer targets are capped and reported below.
Gold labels used for scoring remain untruncated.

This preflight reads compact payloads one at a time without expanding graph features.
Large graph topology is **not** truncated to meet the text budget.


In [ ]:
report = training.token_report(model, dataset, splits["train"] + splits["val"])
print("Training + validation token-limit report:", report)
training.save_and_backup(report, "token-report.json", OUTPUT, DRIVE_OUTPUT)


Training + validation token-limit report: {'answer_targets_truncated': 0, 'descriptions_truncated': 418}


## 7. Train, validate and back up

The best checkpoint is selected by **validation loss**, never test performance.
Early stopping uses validation loss with patience 3.
Correct accumulation, FP16 loss scaling, finite-gradient checks and clipping are enabled.
AMP starts at loss scale 1024. On gradient overflow it lowers the scale and retries
the same accumulation group, restoring BatchNorm statistics and RNG state.
Invalid gradients never update weights or advance the training cursor.
Repeated failure stops after 16 attempts and identifies the affected parameters.
The v2 run names keep this fix separate from older helper fingerprints/checkpoints.

`last.pt` contains adapters, graph modules, optimizer/scaler, RNG and progress.
`best.pt` contains the best adapters and graph modules, including BatchNorm buffers.
Neither contains the base Llama weights. Checkpoints are trusted files from your
own run, loaded in PyTorch's restricted `weights_only` mode.

A single-node training microbatch uses BatchNorm's running statistics with a warning.
The original GAT implementation is reused rather than silently replaced.
Its `GATConv` layers do not configure `edge_dim`; in current PyG, relation features
are not directly included in attention. The text description still contains relations.
This source behavior is another reason not to call this an exact paper reproduction.

**Memory:** fitting every WebQSP retrieved graph on a T4 is not guaranteed.
First reduce text limits if needed. If graph activations are the bottleneck,
changing GNN width changes the experiment and requires a new run name.
Do not silently drop large graphs or treat a smoke run as a full result.


In [ ]:
import time

torch.cuda.reset_peak_memory_stats()
started = time.monotonic()
history = training.fit(model, dataset, splits, SIGNATURE, OUTPUT, DRIVE_OUTPUT)
print(f"Training/resume elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
print("Best validation checkpoint loaded. History:", history)


Backed up optimizer step 50.
Backed up optimizer step 100.
Backed up optimizer step 150.
Backed up optimizer step 200.
Backed up optimizer step 250.
Backed up optimizer step 300.
Backed up optimizer step 350.
Backed up optimizer step 400.
{'epoch': 0, 'train_loss': 2.093523706537065, 'validation_loss': 0.445675679002017, 'optimizer_steps': 415, 'amp_overflows': 0}
Backed up optimizer step 450.
Backed up optimizer step 500.
Backed up optimizer step 550.
Backed up optimizer step 600.
Backed up optimizer step 650.
Backed up optimizer step 700.
Backed up optimizer step 750.
Backed up optimizer step 800.
{'epoch': 1, 'train_loss': 0.28538667377166793, 'validation_loss': 0.4551321543953512, 'optimizer_steps': 830, 'amp_overflows': 0}
Backed up optimizer step 850.
Backed up optimizer step 900.
Backed up optimizer step 950.
Backed up optimizer step 1000.
Backed up optimizer step 1050.
Backed up optimizer step 1100.
Backed up optimizer step 1150.
Backed up optimizer step 1200.
{'epoch': 2, 'tra

## 8. Evaluate the held-out test split

Best validation weights are reloaded before testing.
Raw predictions are backed up every 10 examples and at completion.
A prediction cache is tied to the exact best-checkpoint hash and selected test IDs.

ExplaGraphs reports accuracy and macro-F1 as fractions, plus invalid outputs. The first standalone support/counter word is parsed case-insensitively. Invalid outputs count as wrong. This is stricter than the source substring parser.

The retrieved explanation-graph preprocessing is intentionally different from the authors' original ExplaGraphs baseline.


In [ ]:
predictions, metrics = training.predict_test(
    model, dataset, splits["test"], SIGNATURE, OUTPUT, DRIVE_OUTPUT,
)
print(json.dumps(metrics, indent=2))
print("Outputs are saved locally and backed up to:", DRIVE_OUTPUT)


## 9. Inspect predictions and reuse the trained model

Each prediction retains the input question, full gold label, raw generated text,
processing ID and available source identity.
Older ExplaGraphs snapshots may lack native source IDs; these remain `None`.

The trained object `model` is ready for inference on **preprocessed graphs**.
It is not a text-only classification/QA endpoint: a new question also needs its
own retrieved graph and embeddings from a compatible preprocessing pipeline.

To reload in a fresh Colab session, rerun sections 1-5 with the same configuration,
then call `training.load_best(model, OUTPUT, SIGNATURE)` and `model.eval()`.
The gated chat backbone is downloaded/cached again; adapters and graph modules
come from Drive. Checkpoints are not standalone Llama models.


In [ ]:
for row in predictions[:3]:
    print("Processing ID:", row["id"], "Source ID:", row["source_id"])
    print(row["question"])
    print("Gold:", row["label"])
    print("Prediction:", row["prediction"])
    print("-" * 60)


## 10. Standalone test evaluation after a runtime disconnect

Copy the following **two code cells**, in order, into a fresh Colab notebook.
You do not need to run sections 1-9 or finish all 10 training epochs.
A GPU runtime with available quota is still required; this does not bypass Colab's GPU limit.
Keep `HF_TOKEN` enabled in Colab Secrets and retain access to the gated Llama-2 model.

- Cell A mounts Drive, installs missing dependencies, restores the exact preprocessing snapshot,
  and reads the original settings and test IDs from `run.json`.
- Set `TRAINING_DRIVE` to your existing training folder (the recorded full-v2 path is filled in).
- Helper hashes are checked before loading the model. If only training code changed,
  Cell A finds the saved helper in Git history and verifies that inference code is unchanged.
  Changes to the model, prompting, checkpoint loading, or scoring still stop evaluation.
  If the original helper is not in Git history, use the original training checkout instead.
- Cell B evaluates **best.pt**, selected by validation loss among completed epochs,
  not necessarily the latest epoch. LoRA, GAT, and projector weights are all restored.
  `last.pt` and its optimizer state are not needed for this evaluation.
- Predictions and metrics are saved under **evaluation-best/** inside the training Drive folder.
  Existing training checkpoints and `run.json` are not modified.
  Rerunning these two cells resumes verified prediction backups every 10 examples.

The original pinned 4-bit Llama backbone must still be downloaded in a fresh runtime.
These cells skip training and token-limit reports, not the dependencies needed for inference.


In [ ]:
# Cell A: restore the existing experiment without initializing or resuming training.
from pathlib import Path
import ast
import hashlib
import importlib.util
import json
import subprocess
import sys

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime with available Colab quota before evaluation.")
drive.mount("/content/drive")
TRAINING_DRIVE = Path("/content/drive/MyDrive/GRAG downstream/expla_graphs/expla-graphs-chat-qlora-full-v2")
BACKUP_ROOT = Path("/content/drive/MyDrive/GRAG preprocessing backups")
REPO_URL = "https://github.com/MichaelPham2005/BioGRAG-QA.git"
CODE_REVISION = "GRAG/reproduce/WebQSP"  # Prefer the original training commit if the branch changed.
ROOT = Path("/content/BioGRAG-QA")

for name in ("run.json", "best.pt"):
    if not (TRAINING_DRIVE / name).is_file():
        raise FileNotFoundError(f"Missing {name} in {TRAINING_DRIVE}; check the training folder.")
saved_run = json.loads((TRAINING_DRIVE / "run.json").read_text())
SIGNATURE = saved_run["signature"]
unsigned_run = {key: value for key, value in saved_run.items() if key != "signature"}
if hashlib.sha256(json.dumps(unsigned_run, sort_keys=True).encode()).hexdigest() != SIGNATURE:
    raise ValueError("The saved run.json fingerprint is invalid; restore its original Drive copy.")
if saved_run["format"] != 1 or saved_run["settings"]["dataset"] != "expla_graphs":
    raise ValueError("Select an ExplaGraphs format-1 training run.")

if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", CODE_REVISION], check=True)
helper_paths = {
    "training_helper_sha256": "src/utils/notebook_downstream/training.py",
    "artifact_helper_sha256": "src/utils/notebook_downstream/artifacts.py",
    "gnn_helper_sha256": "src/model/gnn.py",
    "evaluation_helper_sha256": "src/utils/evaluate.py",
}

def evaluation_code(source):
    tree = ast.parse(source)
    training_only = {"initialize_run", "checkpoint_data", "validation_loss",
                     "accumulate_group", "fit", "token_report"}
    tree.body = [node for node in tree.body
                 if not (isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
                         and node.name in training_only)
                 and not (isinstance(node, ast.Import)
                          and all(alias.name == "time" for alias in node.names))
                 and not (isinstance(node, ast.ImportFrom) and node.module == "tqdm.auto")]
    return ast.dump(tree, include_attributes=False)

for key, relative in helper_paths.items():
    path = ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing training helper: {path}")
    current_source = path.read_bytes()
    if hashlib.sha256(current_source).hexdigest() == saved_run[key]:
        continue
    if key != "training_helper_sha256":
        raise ValueError(f"Inference dependency differs: {relative}. Use the original training checkout; "
                         "do not edit run.json or bypass this check.")
    revisions = subprocess.check_output(
        ["git", "-C", str(ROOT), "rev-list", "--all", "--", relative], text=True,
    ).splitlines()
    original_source = None
    for revision in revisions:
        source = subprocess.check_output(["git", "-C", str(ROOT), "show", f"{revision}:{relative}"])
        if hashlib.sha256(source).hexdigest() == saved_run[key]:
            original_source = source
            break
    if original_source is None:
        raise ValueError("The saved training helper was not found in local Git history. "
                         "Use a full clone with the original training commit/source. Do not edit run.json.")
    if evaluation_code(original_source) != evaluation_code(current_source):
        raise ValueError("Model or evaluation code changed since training. Use the original training checkout.")
    print(f"Verified unchanged inference against training commit {revision[:12]}; "
          "training-only changes do not prevent test evaluation.")
requirements = {
    "torch_geometric": "torch-geometric", "transformers": "transformers",
    "accelerate": "accelerate", "bitsandbytes": "bitsandbytes", "peft": "peft",
    "sentencepiece": "sentencepiece", "pandas": "pandas", "tqdm": "tqdm",
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.utils.notebook_downstream import artifacts, training

snapshots = [path.parent for path in sorted(BACKUP_ROOT.glob("*/manifest.json"))
             if artifacts.checksum(path) == saved_run["snapshot_sha256"]]
if len(snapshots) != 1:
    raise ValueError("Expected exactly one preprocessing snapshot matching run.json under BACKUP_ROOT.")
SNAPSHOT = snapshots[0]
PREPARED, PREPROCESS_RUN, snapshot_hash = artifacts.restore_snapshot(
    SNAPSHOT, ROOT / "restored" / "expla_graphs" / SNAPSHOT.name, "expla_graphs",
)
dataset = artifacts.CompactTrainingDataset(PREPARED, PREPROCESS_RUN)
if snapshot_hash != saved_run["snapshot_sha256"] or dataset.config_hash != saved_run["data_config_sha256"]:
    raise ValueError("Restored preprocessing differs from the training run.")
settings = training.Settings(**saved_run["settings"])
splits = saved_run["splits"]
if splits["test"] != dataset.get_idx_split()["test"]:
    raise ValueError("This run did not select the full held-out test split; select the full training run.")
OUTPUT = ROOT / "output/downstream/expla_graphs" / TRAINING_DRIVE.name / "evaluation-best"
DRIVE_OUTPUT = TRAINING_DRIVE / "evaluation-best"
artifacts.restore_training_outputs(DRIVE_OUTPUT, OUTPUT)
for name in ("run.json", "best.pt"):
    artifacts.backup_file(TRAINING_DRIVE / name, OUTPUT / name)
print(f"Restored {len(splits['test'])} test examples; no training will run.")
print("Evaluation output:", DRIVE_OUTPUT)


In [ ]:
# Cell B: load the saved best weights and evaluate only the held-out test split.
from google.colab import userdata
import random
import time

checkpoint = torch.load(OUTPUT / "best.pt", map_location="cpu", weights_only=True)
if checkpoint["signature"] != SIGNATURE:
    raise ValueError("best.pt belongs to a different experiment than run.json.")
print(f"Best saved epoch: {checkpoint['epoch'] + 1}; validation loss: {checkpoint['validation_loss']:.6f}")
del checkpoint
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Enable HF_TOKEN in Colab Secrets with access to the saved Llama model.")
random.seed(settings.seed)
torch.manual_seed(settings.seed)
torch.cuda.manual_seed_all(settings.seed)
model = training.load_qlora(settings, HF_TOKEN)
model.requires_grad_(False)
model.llm.gradient_checkpointing_disable()
started = time.monotonic()
predictions, metrics = training.predict_test(
    model, dataset, splits["test"], SIGNATURE, OUTPUT, DRIVE_OUTPUT,
)
print(json.dumps(metrics, indent=2))
print(f"Test evaluation elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
print("Predictions and metrics backed up to:", DRIVE_OUTPUT)


## 11. Resume the interrupted training run from Drive

In a GPU runtime, run **Cell A in section 10, then the cell below**.
You can skip Cell B until training finishes. No preprocessing or earlier sections are required.
If you already evaluated, this cell releases the inference model and builds a fresh trainable model.

- Resume from **last.pt**, not best.pt: restore LoRA, GAT, projector, optimizer,
  AMP scaler, random states, learning-rate schedule position, and the next saved microbatch.
- Epoch indices 0-6 mean seven completed epochs. A saved partial epoch 7 resumes
  inside the eighth epoch; it is not repeated from the beginning.
- Keep the original **10 total epochs**, not 10 additional epochs.
  Original early stopping is also preserved: with patience 5, training can stop before epoch 10.
  This cell does not silently change patience, settings, or checkpoint fingerprints.
- The exact training helper is recovered from Git history when necessary.
  Your current checkout is not modified, and no weaker evaluation-only compatibility check is used for training.
- Updated last/best checkpoints and history are backed up to the **original training Drive folder**.

After training returns, run **Cell B directly**, without rerunning Cell A.
The cell below prepares a separate evaluation folder keyed by the resulting best-checkpoint hash,
so predictions from your earlier test evaluation are not reused for different weights.

More epochs do not guarantee higher accuracy. Select the checkpoint using validation loss,
not test accuracy. This QLoRA/retrieved-graph setup is not an exact reproduction of the paper.


In [ ]:
# Run section 10 Cell A first; resume the original experiment, not the evaluation folder.
import gc
import time
from google.colab import userdata

if "TRAINING_DRIVE" not in globals() or "snapshot_hash" not in globals():
    raise RuntimeError("Run section 10 Cell A before this continuation cell.")
if not torch.cuda.is_available():
    raise RuntimeError("Training continuation requires an available Colab GPU.")
if saved_run["settings"]["epochs"] != 10:
    raise ValueError("This cell resumes a run originally configured for 10 epochs; do not edit run.json.")
if not (TRAINING_DRIVE / "last.pt").is_file():
    raise FileNotFoundError("last.pt is required to resume optimizer/microbatch state; best.pt is evaluation-only.")

relative = helper_paths["training_helper_sha256"]
RESUME_ROOT = ROOT
if artifacts.checksum(ROOT / relative) != saved_run["training_helper_sha256"]:
    revisions = subprocess.check_output(
        ["git", "-C", str(ROOT), "rev-list", "--all", "--", relative], text=True,
    ).splitlines()
    matching_revision = None
    for revision in revisions:
        source = subprocess.check_output(["git", "-C", str(ROOT), "show", f"{revision}:{relative}"])
        if hashlib.sha256(source).hexdigest() == saved_run["training_helper_sha256"]:
            matching_revision = revision
            break
    if matching_revision is None:
        raise ValueError("Exact training source is missing from Git history; restore the original training checkout.")
    RESUME_ROOT = ROOT.parent / f"BioGRAG-QA-resume-{matching_revision[:12]}"
    if not RESUME_ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "worktree", "add", "--detach",
                        str(RESUME_ROOT), matching_revision], check=True)
for key, relative in helper_paths.items():
    path = RESUME_ROOT / relative
    if not path.is_file() or artifacts.checksum(path) != saved_run[key]:
        raise ValueError(f"Exact training source check failed: {relative}. Original checkpoints were not changed.")
resume_training = artifacts.load_source(
    "colab_resumed_training", RESUME_ROOT / helper_paths["training_helper_sha256"],
)
resume_settings = resume_training.Settings(**saved_run["settings"])
RESUME_OUTPUT = ROOT / "output/downstream/expla_graphs" / TRAINING_DRIVE.name / "resume-training"
artifacts.restore_training_outputs(TRAINING_DRIVE, RESUME_OUTPUT)
last = torch.load(RESUME_OUTPUT / "last.pt", map_location="cpu", weights_only=True)
if last["signature"] != saved_run["signature"]:
    raise ValueError("last.pt belongs to another experiment.")
best = torch.load(RESUME_OUTPUT / "best.pt", map_location="cpu", weights_only=True)
if best["signature"] != saved_run["signature"]:
    raise ValueError("best.pt belongs to another experiment.")
state = last["state"]
best_epoch = best["epoch"] if best["validation_loss"] < state["best_loss"] else state["best_epoch"]
if state["epoch"] >= resume_settings.epochs:
    raise ValueError("This checkpoint already reached the original 10-epoch limit.")
if best_epoch >= 0 and state["epoch"] - 1 - best_epoch >= resume_settings.patience:
    raise ValueError("Original validation early stopping has already triggered; this is not a GPU-interrupted run.")
print(f"Completed epochs: {len(state['history'])}; resuming epoch {state['epoch'] + 1}/10 "
      f"at microbatch {state['next_batch']}; saved optimizer steps: {state['optimizer_steps']}.")
del last, best, state
resume_signature = resume_training.initialize_run(
    resume_settings, dataset, splits, snapshot_hash, RESUME_OUTPUT, TRAINING_DRIVE,
)
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Enable HF_TOKEN in Colab Secrets before loading the model.")
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
model = resume_training.load_qlora(resume_settings, HF_TOKEN)
started = time.monotonic()
history = resume_training.fit(
    model, dataset, splits, resume_signature, RESUME_OUTPUT, TRAINING_DRIVE,
)
print(f"Continuation elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
print(f"Completed epochs: {len(history)}/10 (original validation early stopping preserved).")
del model
gc.collect()
torch.cuda.empty_cache()

evaluation_name = "evaluation-best-" + artifacts.checksum(RESUME_OUTPUT / "best.pt")[:12]
OUTPUT = RESUME_OUTPUT.parent / evaluation_name
DRIVE_OUTPUT = TRAINING_DRIVE / evaluation_name
artifacts.restore_training_outputs(DRIVE_OUTPUT, OUTPUT)
for name in ("run.json", "best.pt"):
    artifacts.backup_file(RESUME_OUTPUT / name, OUTPUT / name)
settings, training, SIGNATURE = resume_settings, resume_training, resume_signature
print("Training checkpoints backed up to:", TRAINING_DRIVE)
print("Next run Cell B directly to evaluate the best checkpoint. Evaluation output:", DRIVE_OUTPUT)
